# 03: Quantificação de Volumes, Calibração SUV e Avaliação de Fibrose

**Objetivo:** Praticar a computação de volume físico real em mililitros ($\text{mL}$), converter o sinal bruto do PET de $\text{Bq/mL}$ para $\text{SUV}_{\text{bw}}$ com correção pelo decaimento radioativo do $^{18}\text{F}$, e avaliar a heterogeneidade metabólica no miocárdio ventricular para identificação de cicatrizes/fibrose.

---

## 1. Importações e Configuração

In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt

import config
from src import io_utils, registration, stats

print("Módulos importados com sucesso!")

Módulos importados com sucesso!


## 2. Carregando Dados e Metadados do Paciente

Vamos carregar o exame completo do paciente `'0001'`.

In [2]:
patient_id = "0001"
caminhos = config.get_caminhos(patient_id)

if caminhos["CT"].exists() and caminhos["PET"].exists():
    ct_img = io_utils.load_sitk_image(caminhos["CT"])
    pet_img = io_utils.load_sitk_image(caminhos["PET"])
    mask_img = io_utils.load_sitk_image(caminhos["Mascara_Cardiaca"]) if caminhos["Mascara_Cardiaca"].exists() else None
    
    # Reamostragem para a grade comum do CT
    reg = registration.full_pet_ct_registration_pipeline(ct_img, pet_img, mask_img, verbose=False)
    
    ct_arr = io_utils.sitk_to_numpy(ct_img)
    pet_bqml = io_utils.sitk_to_numpy(reg["pet_registered"])
    mask_arr = io_utils.sitk_to_numpy(reg["mask_resampled"]) if reg["mask_resampled"] else None
    ct_spacing = io_utils.get_spatial_metadata(ct_img)["spacing"]
    
    print("Dados carregados e reamostrados com sucesso!")
else:
    print("Dados NIfTI não encontrados localmente.")

Dados carregados e reamostrados com sucesso!


## 3. Conversão de Bq/mL para SUVbw

A fórmula clínica de SUV normalizado por peso corporal é:
$$\text{SUV}_{\text{bw}} = \frac{C_{\text{PET}} \text{ [Bq/mL]}}{\frac{\text{Dose Efetiva [Bq]}}{\text{Peso [g]}}}$$

Com decaimento radioativo:
$$\text{Dose}_{\text{aquisição}} = \text{Dose}_{\text{injeção}} \times 2^{-\frac{\Delta t}{109.77}}$$

In [3]:
if config.XLSX_PT_PATH.exists() and 'pet_bqml' in locals():
    df_pt = io_utils.load_pt_details(config.XLSX_PT_PATH)
    meta_pt = io_utils.get_patient_metadata(patient_id, df_pt)
    
    pet_suv, info_suv = io_utils.calculate_suvbw_from_pt_metadata(
        volume_bqml=pet_bqml,
        pt_metadata=meta_pt,
        decay_correction_mode="acquisition"
    )
    
    if info_suv["suv_applied"]:
        print("SUV calculado com sucesso!")
        print(f"  Peso do paciente: {info_suv['weight_kg']} kg")
        print(f"  Dose administrada: {info_suv['dose_mbq']} MBq")
        print(f"  Intervalo injeção-scan: {info_suv.get('intervalo_min', 0):.1f} min")
        print(f"  Fator de decaimento do 18F: {info_suv.get('fator_decaimento', 1.0):.4f}")
        print(f"  Faixa de SUV: [{pet_suv.min():.2f}, {pet_suv.max():.2f}]")
    else:
        print("Metadados insuficientes para SUV. Motivo:", info_suv["reason"])
        pet_suv = pet_bqml

SUV calculado com sucesso!
  Peso do paciente: 91.0 kg
  Dose administrada: 305.0 MBq
  Intervalo injeção-scan: 62.9 min
  Fator de decaimento do 18F: 0.6724
  Faixa de SUV: [0.00, 64.27]


## 4. Estatísticas de VOI e Volume Real em mL

Vamos calcular o volume anatômico em $\text{mL}$ e a captação média/máxima para cada estrutura cardíaca segmentada na máscara.

In [5]:
if 'mask_arr' in locals() and mask_arr is not None:
    tabela_stats = stats.cardiac_structures_statistics(
        ct_volume=ct_arr,
        pet_volume=pet_suv,
        mask_volume=mask_arr,
        cardiac_labels=config.ROTULOS_CARDIACOS,
        spacing_mm=ct_spacing
    )
    
    print("=" * 75)
    print(f"{'Estrutura':<25s} | {'Volume (mL)':<12s} | {'SUV Médio':<10s} | {'SUV Máx':<10s} | {'HU Médio':<10s}")
    print("-" * 75)
    for lbl, st in tabela_stats.items():
        nome = st['label_name']
        vol = st.get('volume_ml', 0.0)
        s_mean = st['PET_SUV']['mean']
        s_max = st['PET_SUV']['max']
        ct_mean = st['CT_HU']['mean']
        print(f"{nome:<25s} | {vol:10.2f} mL | {s_mean:10.2f} | {s_max:10.2f} | {ct_mean:10.1f}")
    print("=" * 75)


  Estatísticas por Estrutura Cardíaca — PET/CT Conjunto
  [ 1] heart_myocardium             vol= 127.0mL | SUV méd=10.47 (máx=25.11) | HU méd=  83.3
  [ 2] heart_atrium_left            vol=  80.5mL | SUV méd= 2.96 (máx=12.21) | HU méd= 156.2
  [ 3] heart_atrium_right           vol=  80.4mL | SUV méd= 2.48 (máx= 4.26) | HU méd= 125.5
  [ 4] heart_ventricle_left         vol= 110.2mL | SUV méd= 6.72 (máx=21.85) | HU méd= 141.4
  [ 5] heart_ventricle_right        vol= 149.5mL | SUV méd= 2.65 (máx= 8.58) | HU méd= 120.5
  [ 6] aorta                        vol= 235.0mL | SUV méd= 2.58 (máx= 4.96) | HU méd= 163.1
  [ 7] iliac_artery_left            vol=  17.1mL | SUV méd= 2.37 (máx= 3.99) | HU méd= 134.3
  [ 8] iliac_artery_right           vol=  16.8mL | SUV méd= 2.40 (máx= 4.93) | HU méd= 147.4
  [ 9] iliac_vena_left              vol=  29.6mL | SUV méd= 2.12 (máx= 4.10) | HU méd= 107.9
  [10] iliac_vena_right             vol=  29.3mL | SUV méd= 2.13 (máx= 3.84) | HU méd= 118.2
  [11] inferi

## 5. Investigação de Fibrose no Miocárdio (Label 1)

Avaliamos o Coeficiente de Variação ($CV = \sigma / \mu$) no miocárdio para verificar heterogeneidade focal de captação.

In [6]:
if 'tabela_stats' in locals() and config.LABEL_MIOCARDIO in tabela_stats:
    m = tabela_stats[config.LABEL_MIOCARDIO]
    media = m['PET_SUV']['mean']
    desvio = m['PET_SUV']['std']
    cv = desvio / (media + 1e-6)
    
    print("Avaliação de Fibrose no Miocárdio:")
    print(f"  Captação Média: {media:.2f}")
    print(f"  Desvio Padrão:  {desvio:.2f}")
    print(f"  Coeficiente de Variação (CV): {cv:.2%}")
    
    if cv > 0.40:
        print("  -> ALERTA: Heterogeneidade elevada (> 40%). Investigar área focal de fibrose/isquemia!")
    else:
        print("  -> Padrão metabólico uniforme e homogêneo.")

Avaliação de Fibrose no Miocárdio:
  Captação Média: 10.47
  Desvio Padrão:  3.43
  Coeficiente de Variação (CV): 32.77%
  -> Padrão metabólico uniforme e homogêneo.
